# Scripts for measuring noise on a large skypatch image

In [ ]:
# | default_exp euclid.skypatch_noise

In [ ]:
# | exporti

import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.stats import sigma_clip
from astropy.visualization import simple_norm
from photutils.aperture import CircularAnnulus, EllipticalAnnulus, RectangularAperture
from scipy.stats import median_abs_deviation

In [ ]:
# | exporti

plt.style.use("nicl.euclid.v1nicl")

In [ ]:
# | export


def noise_profile_diagnostics(
    extended_measurement_table,
    label=None,
    output_path=None,
    zp=23.9,
):
    output_path = Path(output_path)
    grouped = extended_measurement_table.groupby("Centre_pixel")

    def plot_profiles(ax, x, y, ylabel):
        ax.plot(x, y, marker="o", markersize=2)
        ax.hlines(0, xmin=np.min(x), xmax=np.max(x), color="k", lw=1, ls="-")
        ax.set_ylabel(ylabel, fontsize=14)
        ax.set_xlabel("Radius (arcsec)", fontsize=14)
        ax.set_ylim(-0.2, 0.2)

    # Plot: Clipped median & mean flux profiles
    fig, ax = plt.subplots(1, 2, figsize=(8, 4))
    for _, group in grouped:
        plot_profiles(
            ax[0],
            group["SMA_annulus_centre_arcsec"],
            group["Clipped_median_flux_aperture"],
            "Clipped Median Flux",
        )
        plot_profiles(
            ax[1],
            group["SMA_annulus_centre_arcsec"],
            group["Clipped_mean_flux_aperture"],
            "Clipped Mean Flux",
        )
    fig.tight_layout()
    if output_path:
        plt.savefig(output_path / f"{label}_Median_Mean_Flux_CLIPPED.pdf")
        plt.close()
    else:
        plt.show()

    # Plot: Background-subtracted flux profiles
    fig, ax = plt.subplots(figsize=(4, 4))
    for _, group in grouped:
        ax.plot(
            group["SMA_annulus_centre_arcsec"],
            group["bkg_sub_flux"],
            marker="o",
            markersize=2,
        )
        ax.hlines(
            group["bkg_noise"].iloc[0],
            xmin=np.min(group["SMA_annulus_centre_arcsec"]),
            xmax=np.max(group["SMA_annulus_centre_arcsec"]),
            color="k",
            lw=0.2,
            ls="--",
        )
    ax.hlines(
        0,
        xmin=0,
        xmax=np.max(extended_measurement_table["SMA_annulus_centre_arcsec"]),
        color="k",
        lw=0.5,
        ls="-",
    )
    ax.set_ylabel("Background-subtracted Median Flux", fontsize=12)
    ax.set_xlabel("Radius (arcsec)", fontsize=12)
    fig.tight_layout()
    if output_path:
        plt.savefig(output_path / f"{label}_BkgsubFlux.pdf")
        plt.close()
    else:
        plt.show()

    # Plot: MAD profiles in flux and magnitude space
    flux_grouped_rad = extended_measurement_table.groupby("SMA_annulus_centre_arcsec")
    radii = sorted(flux_grouped_rad.groups.keys())

    mad_median_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["Median_flux_aperture"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]
    mad_clipped_median_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["Clipped_median_flux_aperture"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]
    mad_bkg_sub_flux = [
        median_abs_deviation(
            flux_grouped_rad.get_group(r)["bkg_sub_flux"],
            scale="normal",
            nan_policy="omit",
        )
        for r in radii
    ]

    fig, ax = plt.subplots(1, 2, figsize=(8, 4))

    # Plot in flux units
    ax[0].plot(radii, np.array(mad_median_flux) * 3, label="MAD of Median Flux")
    ax[0].plot(
        radii, np.array(mad_clipped_median_flux) * 3, label="MAD of Clipped Median Flux"
    )
    ax[0].plot(
        radii, np.array(mad_bkg_sub_flux) * 3, label="MAD of Bkg Subtracted Flux"
    )
    ax[0].set_ylabel("3xMAD (Flux / arcsec²)", fontsize=14)
    # ax[0].set_xscale("log")
    ax[0].grid(True)
    ax[0].legend()
    ax[0].set_ylim(-0.2, 2)

    # Plot in magnitude units
    def flux_to_mag(flux):
        return zp - 2.5 * np.log10(flux)

    ax[1].plot(
        radii, flux_to_mag(np.array(mad_median_flux) * 3), label="MAD of Median Flux"
    )
    ax[1].plot(
        radii,
        flux_to_mag(np.array(mad_clipped_median_flux) * 3),
        label="MAD of Clipped Median Flux",
    )
    ax[1].plot(
        radii,
        flux_to_mag(np.array(mad_bkg_sub_flux) * 3),
        label="MAD of Bkg Subtracted Flux",
    )
    ax[1].invert_yaxis()
    ax[1].set_ylabel(r"3×MAD [mag / $\rm arcsec^2$]", fontsize=14)
    # ax[1].set_xscale("log")
    ax[1].grid(True)
    ax[1].legend()

    for a in ax:
        a.set_xlabel("Radius (arcsec)", fontsize=14)

    fig.tight_layout()
    if output_path:
        plt.savefig(output_path / f"{label}_MAD_profiles.pdf")
        plt.close()
    else:
        plt.show()

In [ ]:
# | export


def measure_noise_in_apertures(
    image_path,  # path to the image file for measurement
    mask_path,  # path to the mask file
    aperture_type="annuli",  # 'annuli' or 'boxes'
    r_max=3000,  # maximum radius/half-size in pixels
    gscale=0.4,  # geometric scale by which to grow annuli/boxes
    ellipticity=0.5,  # ellipticity to use for all elliptical annuli (1-b/a), ignored for boxes/circles
    bkg_radius=1000,  # radius/half-size beyond which to measure background
    num_points=1000,  # number of points with which to measure noise
    pixelscale=0.3,  # pixelscale of the image in arcsec/pixel
    label=None,  # label for the output files
    output_path=None,  # path to save output
    plot_overlay=False,  # save overlay plot of apertures/boxes
    save_diagnostics=False,  # save diagnostic plots
    zp=23.9,  # zero point of the image
):
    """Measure noise in annular or square (box) apertures for a given image and mask."""
    log = logging.getLogger(__name__)
    log.info(f"Requested rmax={r_max:.1f} pixels.")

    n_steps = int(np.floor(np.log(r_max) / np.log(1 + gscale))) + 1
    final_r_max = (1 + gscale) ** n_steps

    if aperture_type == "annuli":
        radii = np.geomspace(1, final_r_max, n_steps + 1)
        step_descriptor = f"annuli with ellipticity={ellipticity:.3f}, rmax={final_r_max:.1f} pixels and gscale={gscale:.3f}."
        edge_margin = final_r_max
    elif aperture_type == "boxes":
        box_sizes = np.geomspace(1, final_r_max, n_steps + 1)
        # force box sizes to be odd integer side lengths
        box_sizes = np.ceil(box_sizes).astype(int)
        box_sizes[box_sizes % 2 == 0] += 1
        box_sizes = np.unique(box_sizes)
        n_steps = len(box_sizes)
        step_descriptor = f"square boxes, max half-size={final_r_max:.1f} pixels, gscale={gscale:.3f}."
        edge_margin = box_sizes[-1] // 2
    else:
        raise ValueError("aperture_type must be 'annuli' or 'boxes'.")
    log.info(f"Measuring noise in {n_steps} {aperture_type}: {step_descriptor}")

    image = fits.getdata(image_path)
    mask = fits.getdata(mask_path).astype(bool)
    image = np.where(mask, np.nan, image)
    mask = ~np.isfinite(image)

    image_height, image_width = image.shape
    valid_points = np.where(~mask)
    valid_points = np.column_stack((valid_points[1], valid_points[0]))

    valid_points = valid_points[
        (valid_points[:, 0] > edge_margin)
        & (valid_points[:, 0] < image_width - edge_margin)
        & (valid_points[:, 1] > edge_margin)
        & (valid_points[:, 1] < image_height - edge_margin)
    ]
    if valid_points.size < 2:
        log.error("Not enough valid points found for noise measurement.")
        return

    centres = []
    pas = []
    flux_stats = []
    np.random.seed(42)  # pick the same random points for consistency
    while len(centres) < num_points:
        log.debug(f"Centre {len(centres) + 1} of {num_points}")
        centre = valid_points[np.random.choice(len(valid_points))]
        if aperture_type == "boxes":
            # Must be pixel-centred
            centre = (round(centre[0]), round(centre[1]))
            pa = 0.0
        else:
            centre = centre + np.random.uniform(-0.5, 0.5, 2)
            centre = tuple(centre)
            pa = np.random.uniform(0, 180)  # in degrees, random PA for each point

        pas.append(pa)
        centres.append(centre)
        log.debug(
            f"Extracting noise profile around point: ({centre[0]:.1f}, {centre[1]:.1f}), PA={pa:.1f}"
        )

        for i in range(n_steps):
            if aperture_type == "annuli":
                sma_in, sma_out = radii[i], radii[i + 1]
                ba = 1.0 - ellipticity
                theta_rad = np.deg2rad(pa)
                if np.isclose(ba, 0.0):
                    aperture = CircularAnnulus(centre, sma_in, sma_out)
                else:
                    aperture = EllipticalAnnulus(
                        centre,
                        a_in=sma_in,
                        a_out=sma_out,
                        b_out=sma_out * ba,
                        b_in=sma_in * ba,
                        theta=theta_rad,
                    )
            elif aperture_type == "boxes":
                side = box_sizes[i]
                # Always centered on a pixel and not rotated
                aperture = RectangularAperture(
                    centre,
                    w=side,
                    h=side,
                )
            else:
                raise ValueError("Unknown aperture_type.")

            aperture_mask = aperture.to_mask(method="center")
            values = aperture_mask.get_values(image, mask)

            if aperture_type == "annuli":
                sma = (sma_in + sma_out) / 2
            else:
                sma = side / 2  # for "radius" proxy (half-side in pixels)

            stats = {
                "Centre_pixel": centre,
                "PA_deg": pa,
                "Ellipticity": ellipticity if aperture_type == "annuli" else 0.0,
            }
            if aperture_type == "annuli":
                stats.update(
                    {
                        "Inner_Radius_pix": sma_in,
                        "Outer_Radius_pix": sma_out,
                        "Inner_Radius_arcsec": sma_in * pixelscale,
                        "Outer_Radius_arcsec": sma_out * pixelscale,
                        "SMA_annulus_centre_pix": sma,
                        "SMA_annulus_centre_arcsec": sma * pixelscale,
                    }
                )
            else:
                stats.update(
                    {
                        "Box_halfside_pix": sma,
                        "Box_halfside_arcsec": sma * pixelscale,
                    }
                )

            if values is None or len(values) < 1:
                stats.update(
                    {
                        key: np.nan
                        for key in [
                            "Mean_flux_aperture",
                            "Median_flux_aperture",
                            "Std_flux_aperture",
                            "Clipped_mean_flux_aperture",
                            "Clipped_median_flux_aperture",
                            "Clipped_Std_flux_aperture",
                            "Total_valid_pix_aperture",
                        ]
                    }
                )
            else:
                if len(values) < 5:
                    clipped_values = values
                else:
                    clipped = sigma_clip(values, sigma=3, cenfunc="median", maxiters=5)
                    clipped_values = clipped.data[~clipped.mask]
                # rescale such that it's flux per arcsec^2 for all cases
                stats.update(
                    {
                        "Mean_flux_aperture": np.nanmean(values) / pixelscale**2,
                        "Median_flux_aperture": np.nanmedian(values) / pixelscale**2,
                        "Std_flux_aperture": np.nanstd(values),
                        "Clipped_mean_flux_aperture": np.nanmean(clipped_values)
                        / pixelscale**2,
                        "Clipped_median_flux_aperture": np.nanmedian(clipped_values)
                        / pixelscale**2,
                        "Clipped_Std_flux_aperture": np.nanstd(clipped_values),
                        "Total_valid_pix_aperture": len(values),
                    }
                )

            flux_stats.append(stats)

    if len(flux_stats) == 0:
        log.error("No valid points found for noise measurement.")
        return

    flux_stats = pd.DataFrame(flux_stats)

    # Background subtraction
    if aperture_type == "annuli":
        group_key = "SMA_annulus_centre_pix"
        arcsec_key = "SMA_annulus_centre_arcsec"
    else:
        group_key = "Box_halfside_pix"
        arcsec_key = "Box_halfside_arcsec"

    flux_grouped = flux_stats.groupby("Centre_pixel")
    flux_combined = []
    for _, group in flux_grouped:
        group = group.copy()
        bkg_region = group[group[group_key] > bkg_radius]
        if bkg_region.empty:
            group["bkg_sub_flux"] = group["bkg_noise"] = np.nan
        else:
            bkg = np.nanmean(bkg_region["Clipped_median_flux_aperture"])
            noise = np.nanstd(bkg_region["Clipped_median_flux_aperture"])
            group["bkg_sub_flux"] = group["Clipped_median_flux_aperture"] - bkg
            group["bkg_noise"] = 3 * noise
        flux_combined.append(group)

    extended_measurement_table = pd.concat(flux_combined, ignore_index=True)

    # Median Absolute Deviation calculation as noise
    mad_group = extended_measurement_table.groupby(arcsec_key)
    radii = np.array(sorted(mad_group.groups.keys()))
    noise_measurements = pd.DataFrame(
        {
            group_key: radii / pixelscale,
            arcsec_key: radii,
            "MAD_Median_Clipped_Flux": [
                median_abs_deviation(
                    mad_group.get_group(r)["Clipped_median_flux_aperture"],
                    scale="normal",
                    nan_policy="omit",
                )
                for r in radii
            ],
            "MAD_Bkg_Subtracted_Flux": [
                median_abs_deviation(
                    mad_group.get_group(r)["bkg_sub_flux"],
                    scale="normal",
                    nan_policy="omit",
                )
                for r in radii
            ],
            "Median_Valid_Pixels": [
                np.nanmedian(mad_group.get_group(r)["Total_valid_pix_aperture"])
                for r in radii
            ],
        }
    )

    if output_path:
        output_path = Path(output_path)
        output_path.mkdir(parents=True, exist_ok=True)
        if aperture_type == "annuli":
            fn = "noise_stats_extended_datatable.csv"
        else:
            fn = "noise_stats_extended_datatable_boxes.csv"
        if label is not None:
            fn = f"{label}_{fn}"
        extended_measurement_table.to_csv(output_path / fn, index=False)
        if aperture_type == "annuli":
            fn = "noise_measurements.csv"
        else:
            fn = "noise_measurements_boxes.csv"
        if label is not None:
            fn = f"{label}_{fn}"
        noise_measurements.to_csv(output_path / fn, index=False)

    if plot_overlay:
        log.info(
            f"Plotting the {aperture_type} overlays on the image... May take a while if your image is big, and/or if you have lots of apertures... :) "
        )
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(
            image,
            origin="lower",
            cmap="gray",
            norm=simple_norm(image, "sqrt", percent=99),
        )
        ax.set_title(f"Overlay of All Fitted {aperture_type.title()}s")

        for i, coord in enumerate(centres):
            x, y = coord
            pa = pas[i]
            if aperture_type == "annuli":
                ba = 1.0 - ellipticity
                for j in range(n_steps):
                    sma = (radii[j] + radii[j + 1]) / 2
                    if np.isclose(ba, 0.0):
                        ring = plt.Circle(
                            (x, y),
                            radius=sma,
                            edgecolor="magenta",
                            facecolor="none",
                            linewidth=0.4,
                            alpha=0.5,
                        )
                        ax.add_patch(ring)
                    else:
                        e2 = plt.matplotlib.patches.Ellipse(
                            (x, y),
                            width=2 * sma,
                            height=2 * sma * ba,
                            angle=pa,
                            edgecolor="magenta",
                            facecolor="none",
                            linewidth=0.4,
                            alpha=0.5,
                        )
                        ax.add_patch(e2)
            elif aperture_type == "boxes":
                for j in range(n_steps):
                    side = box_sizes[j]
                    rect = plt.matplotlib.patches.Rectangle(
                        (x - side // 2, y - side // 2),
                        width=side,
                        height=side,
                        angle=0.0,
                        edgecolor="magenta",
                        facecolor="none",
                        linewidth=0.4,
                        alpha=0.5,
                    )
                    ax.add_patch(rect)
        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(0, image.shape[0])
        fn = f"{label}_{aperture_type}.pdf"
        plt.savefig(output_path / fn)
        plt.close()

    if save_diagnostics:
        if aperture_type == "annuli":
            diagnostics_path = output_path / "annuli_diagnostics"
        else:
            diagnostics_path = output_path / "boxes_diagnostics"
        diagnostics_path.mkdir(parents=True, exist_ok=True)
        noise_profile_diagnostics(
            extended_measurement_table,
            label=label,
            output_path=diagnostics_path,
            zp=zp,
        )

    return extended_measurement_table, noise_measurements


def measure_noise_in_annuli(*args, **kwargs):
    """Measure noise in elliptical or circular annuli."""
    if "plot_annuli" in kwargs:
        kwargs["plot_overlay"] = kwargs.pop("plot_annuli")
    return measure_noise_in_apertures(aperture_type="annuli", *args, **kwargs)


def measure_noise_in_boxes(*args, **kwargs):
    """Measure noise in square boxes."""
    return measure_noise_in_apertures(aperture_type="boxes", *args, **kwargs)

In [ ]:
# | export


def sb_limit_q1(
    r: float | np.ndarray,  # annular aperture radius in arcsec
    boxsize: float | np.ndarray,  # background subtraction boxsize in pixels
    band: str | None = None,  # band, if None, must provide params
    field: str | None = None,  # field, if None, must provide params
    params: list | None = None,  # if not None, use these instead of hardcoded values
) -> float | np.ndarray:  # 3-sigma surface brightness limit in mag/arcsec^2
    """Calculate the 3-sigma surface brightness limit for a given annulus radius, background subtraction boxsize, band, and field.

    Uses a smooth fit to the measured noise, with an RMS of 0.003 mag/arcsec^2.

    """
    log = logging.getLogger(__name__)
    r = np.asarray(r)
    boxsize = np.asarray(boxsize)
    if ((r < 1) | (r > 1000)).any():
        log.warning("r is out of constrained range [1, 1000] arcsec")
    if ((boxsize < 300) | (boxsize > 2750)).any():
        log.warning("boxsize is out of constrained range [300, 2750] pixels")

    # Two-parameter fit (rms=0.035)
    # noise_function = "((#1 ^ 0.16171958) * 3.6932898) - (((((0.75049275 ^ #1) + (#6 * #6)) * #3) + ((((6.629865 ^ #1) + ((#1 ^ #6) * 3.8938165)) / ((#2 / 1.9729091) + ((((#1 ^ (#6 * 3.4717138)) + (6.124605 ^ #1)) * 3.469273) + (-22.269053 / #3)))) * #5)) + -22.26995)"
    # band_param1 = [-0.25571358, -0.3181491, -1.3309591, -0.2751939]
    # band_param2 = [-0.73960793, 0.14014645, -0.20889468, 0.059540708]
    # field_param1 = [-9.638679, -8.59512, -9.013615]
    # field_param2 = [0.7569607, -0.5547611, 0.7604514]

    # Three-parameter fit to original set of noise curves and missing YJH (rms=0.0023)
    # noise_function = "(((#1 - #4) * (#8 ^ 1.3912746)) - ((((-0.00074630475 / #6) + #5) + (#1 * (#1 - (#5 * #4)))) * ((#7 * ((#4 * 0.25506523) + ((#1 + (260.185 / (#2 + ((#1 ^ (#1 ^ (#1 / #8))) - #3)))) - (#2 * 0.00010827427)))) + 1.1077273))) + 26.196531"
    # band_param1 = [-0.00811447, -0.20032445, -3.1353347, 0.08817829]
    # band_param2 = [0.4924656, 0.4457736, -0.5701219, 0.47016442]
    # band_param3 = [-0.017863939, -0.012539046, 0.54210037, 0.04041138]
    # field_param1 = [-0.022179797, -2.4409833, 0.022864059]
    # field_param2 = [-0.25712505, -0.27963218, -0.25830427]
    # field_param3 = [1.7340696, 1.5605289, 1.7003783]

    # Three-parameter fit to extended set of noise curves and including YJH (rms=0.0029)
    noise_function = "#7 + (((#7 + #1) * ((#3 - #6) + (((#5 / #2) + (0.24884821 ^ ((#1 * #1) ^ #8))) ^ 0.39128637))) + ((#4 + #5) * ((((((0.16409773 ^ #1) ^ (#5 / #2)) * 3.6052904) ^ #1) / (#2 + (#5 * #4))) + #8)))"
    band_param1 = [0.42597696, 0.40886882, 0.21826088, 0.43058214, 0.34954134]
    band_param2 = [0.64209265, 0.034851216, -1.2052855, 0.010148207, 1.0220108]
    band_param3 = [24.680067, 25.407665, 28.408003, 25.314537, 24.841537]
    field_param1 = [-0.48278528, -0.558892, -0.44869965]
    field_param2 = [0.4612925, 1.174292, 0.5953054]
    field_param3 = [0.93746895, 0.8562339, 0.92665035]

    band_param1 = dict(zip(["H", "J", "VIS", "Y", "YJH"], band_param1))
    band_param2 = dict(zip(["H", "J", "VIS", "Y", "YJH"], band_param2))
    band_param3 = dict(zip(["H", "J", "VIS", "Y", "YJH"], band_param3))
    field_param1 = dict(zip(["EDFF", "EDFN", "EDFS"], field_param1))
    field_param2 = dict(zip(["EDFF", "EDFN", "EDFS"], field_param2))
    field_param3 = dict(zip(["EDFF", "EDFN", "EDFS"], field_param3))

    if params is None:
        b1 = band_param1[band]  # noqa: F841
        b2 = band_param2[band]  # noqa: F841
        b3 = band_param3[band]  # noqa: F841
        f1 = field_param1[field]  # noqa: F841
        f2 = field_param2[field]  # noqa: F841
        f3 = field_param3[field]  # noqa: F841
    else:
        b1, b2, b3, f1, f2, f3 = params

    r = np.log10(r)  # noqa: F841
    s = boxsize  # noqa: F841
    variables = ["r", "s", "b1", "b2", "b3", "f1", "f2", "f3"]
    for i, variable in enumerate(variables):
        noise_function = noise_function.replace(f"#{i + 1}", variable)
    noise_function = noise_function.replace("^", "**")
    noise = eval(noise_function)
    noise = np.squeeze(noise)[()]
    return noise


def bkg_err_q1(
    r: float | np.ndarray,  # annular aperture radius in arcsec
    boxsize: float | np.ndarray,  # background subtraction boxsize in pixels
    band: str | None = None,  # band, if None, must provide params
    field: str | None = None,  # field, if None, must provide params
    params: list | None = None,  # if not None, use these instead of hardcoded values
    zp=23.9,  # default to the zeropoint used in our combined images
) -> float | np.ndarray:  # 1-sigma background error in flux/arcsec^2
    """Calculate the 1-sigma background error for a given annulus radius, background subtraction boxsize, band, and field.

    Uses a smooth fit to the measured 3-sigma surface brightness limit, with an RMS of 0.003 mag/arcsec^2,
    and converts to the 1-sigma error in the background level in flux/arcsec^2.

    """
    sb_lim = sb_limit_q1(r, boxsize, band, field, params)
    flux_err = 10 ** (-0.4 * (sb_lim - zp)) / 3
    return flux_err